# BDD100K — Task 3: Evaluation & Visualization

**Bosch Applied CV Assignment | Notebook 03**

This notebook covers:
1. Evaluation metrics selection and rationale
2. Quantitative evaluation (mAP@0.5, Precision, Recall, per-class AP)
3. Confusion matrix analysis
4. Precision-Recall curves
5. Qualitative visualization (GT vs Prediction overlays)
6. Failure case gallery
7. Failure cluster analysis (by scene, weather, time-of-day)
8. Model improvement suggestions

In [1]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import cv2

from src.data_analysis.bdd_parser import BDDDataset, BDD_DETECTION_CLASSES
from src.model.model_loader import YOLOv8Detector
from src.evaluation.evaluator import BDDEvaluator
from src.evaluation.visualizer import (
    plot_map_barchart,
    plot_precision_recall_curves,
    plot_confusion_matrix,
    visualize_predictions,
    plot_failure_gallery,
    plot_failure_clusters,
    plot_missed_object_sizes,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120
print('Imports OK')

Imports OK


## 1. Evaluation Metrics — Selection & Rationale

| Metric | Formula | Why Chosen |
|--------|---------|------------|
| **mAP@0.5** | Mean of per-class AP at IoU≥0.5 | Industry standard; primary leaderboard metric for BDD100K |
| **mAP@0.5:0.95** | Mean of mAP across IoU thresholds 0.5→0.95 | COCO-style; punishes poor localisation — important for safety |
| **Precision** | TP/(TP+FP) | AV systems must avoid false alarms (phantom braking is dangerous) |
| **Recall** | TP/(TP+FN) | Missing a pedestrian or cyclist is a safety-critical failure |
| **Per-class AP** | AP per class | Reveals which object types the model handles poorly |
| **Confusion Matrix** | Predicted vs GT class grid | Shows class-pair confusions (e.g., rider ↔ pedestrian) |

**Autonomous driving priority:** Recall > Precision (it's safer to false-alarm than to miss a real pedestrian).

In [2]:
# ── Update these paths ──
TRAIN_JSON = '../Data/labels/bdd100k_labels_images_train.json'
VAL_JSON   = '../Data/labels/bdd100k_labels_images_val.json'
TRAIN_IMAGE_DIR = '../Data/images/train'
VAL_IMAGE_DIR   = '../Data/images/val'
OUTPUT_DIR   = '../results/evaluation'

os.makedirs(OUTPUT_DIR, exist_ok=True)

## 2. Load Dataset & Model

In [3]:
# Load annotations
dataset = BDDDataset(train_json=TRAIN_JSON, val_json=VAL_JSON, image_dir='../Data/images')
dataset.load()
val_frames = dataset.val_frames
print(f'Validation frames: {len(val_frames):,}')

[BDDDataset] Loading training annotations from: ../Data/labels/bdd100k_labels_images_train.json
[BDDDataset] Loaded 69863 training frames.
[BDDDataset] Loading validation annotations from: ../Data/labels/bdd100k_labels_images_val.json
[BDDDataset] Loaded 10000 validation frames.
Validation frames: 10,000


In [ ]:
# Load YOLOv8n detector
detector = YOLOv8Detector(
    model_name='yolov8n.pt',
    conf_threshold=0.25,
    device='cpu',  # Change to 'cuda' if available
)
detector.load()

## 3. Quantitative Evaluation

In [ ]:
# Run evaluation
# max_images=200 for quick run; remove for full val set (10k images)
evaluator = BDDEvaluator(
    detector=detector,
    val_frames=val_frames,
    val_image_dir=VAL_IMAGE_DIR,
    iou_threshold=0.5,
)
results = evaluator.evaluate(max_images=200)  # Remove limit for full evaluation
evaluator.print_report(results)

In [ ]:
# Save results to CSV and JSON
evaluator.save_results(results, OUTPUT_DIR)

# Show as DataFrame
metrics_df = pd.read_csv(os.path.join(OUTPUT_DIR, 'per_class_metrics.csv'))
metrics_df.style.background_gradient(cmap='RdYlGn', subset=['AP@0.5', 'Recall']).format('{:.3f}', subset=['AP@0.5', 'Precision', 'Recall'])

## 4. AP Bar Chart

In [ ]:
plot_map_barchart(
    per_class_ap=results['per_class_ap'],
    mAP_50=results['mAP_50'],
    output_dir=None,  # None = show inline
)

## 5. Confusion Matrix

In [ ]:
plot_confusion_matrix(
    confusion_matrix=results['confusion_matrix'],
    output_dir=None,
)

**Analysis of Confusion Matrix:**
- `pedestrian` ↔ `rider`: High confusion expected (both are persons with different context)
- `car` → `truck`: Cars sometimes classified as trucks (similar shape, different scale)
- `traffic light` ↔ `traffic sign`: Visual similarity at distance

## 6. Qualitative Visualization — GT vs Predictions

In [ ]:
# Run full inference on a subset for qualitative analysis
EVAL_FRAMES = val_frames[:100]  # Use more for comprehensive analysis

image_paths = [
    os.path.join(VAL_IMAGE_DIR, f.name)
    for f in EVAL_FRAMES
    if os.path.exists(os.path.join(VAL_IMAGE_DIR, f.name))
]
print(f'Running inference on {len(image_paths)} images ...')
all_detections = detector.predict_batch(image_paths, batch_size=8)
print(f'Total detections: {sum(len(v) for v in all_detections.values()):,}')

In [ ]:
# Save overlay images
overlay_dir = os.path.join(OUTPUT_DIR, 'overlays')
visualize_predictions(
    frames=EVAL_FRAMES,
    all_detections=all_detections,
    image_dir=VAL_IMAGE_DIR,
    output_dir=overlay_dir,
    n_samples=10,
)

In [ ]:
# Show one overlay inline
overlay_files = [f for f in os.listdir(overlay_dir) if f.endswith('.jpg')]
if overlay_files:
    img = cv2.imread(os.path.join(overlay_dir, overlay_files[0]))
    plt.figure(figsize=(16, 8))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title(f'GT (thick) vs Predictions (thin) — {overlay_files[0]}')
    plt.show()

## 7. Failure Case Gallery

In [ ]:
failure_dir = os.path.join(OUTPUT_DIR, 'failures')
failure_df = plot_failure_gallery(
    frames=EVAL_FRAMES,
    all_detections=all_detections,
    image_dir=VAL_IMAGE_DIR,
    output_dir=failure_dir,
    top_n=10,
)

print('\nTop 10 worst frames (highest miss rate):')
failure_df.head(10)[['name', 'gt_count', 'missed', 'score', 'timeofday', 'weather', 'scene']]

## 8. Failure Cluster Analysis

In [ ]:
plot_failure_clusters(failure_df, output_dir=None)

In [ ]:
# Missed object size analysis
plot_missed_object_sizes(
    frames=EVAL_FRAMES,
    all_detections=all_detections,
    output_dir=None,
)

## 9. Personal Analysis — What Works & What Doesn't

### What Works Well
- **Large vehicles (bus, truck, car)**: High AP due to large bounding boxes, distinctive shape and texture
- **Traffic signs and lights**: Distinctive colours (red/green/yellow) and shapes help
- **Daytime scenes**: Well-lit images allow good feature extraction

### What Doesn't Work Well
- **Small objects (rider, bicycle, pedestrian at distance)**: Sub-30px boxes are below the effective receptive field of P3 detection head
- **Night-time scenes**: Low contrast; pre-trained COCO model has limited night-time data
- **Occluded pedestrians in crowds**: Partial visibility leads to FN
- **Rare classes (train)**: Very few training examples → low AP
- **Rider vs Pedestrian confusion**: Contextual difference (on bike) is hard to resolve without bicycle detected nearby

### Suggested Improvements

| Issue | Fix |
|-------|-----|
| Night failures | Add night augmentation (gamma darkening, brightness jitter) |
| Small objects | Use higher resolution (1280×1280) or SAHI (Slicing Aided Hyper Inference) |
| Class imbalance | Use focal loss or class-weighted sampling (upsample rare classes) |
| Occlusion | Add mosaic + mixup augmentation (both enabled by default in YOLOv8) |
| Rider confusion | Train with `rider` + `bicycle` context; consider multi-label or relation modeling |
| Low recall on train | Collect more train-class images or use synthetic augmentation |

### Connection to Data Analysis
- EDA showed **40%+ night-time frames** → failure cluster confirms night is hardest
- EDA showed **car dominates 65%+ of instances** → model over-predicts cars at expense of rare classes
- EDA anomalies (empty frames, occluded objects) → directly explain FP and FN clusters seen here

In [ ]:
# Final summary DataFrame
print('\n===== FINAL EVALUATION SUMMARY =====')
print(f"mAP@0.5 : {results['mAP_50']:.3f}")
print('\nPer-Class AP:')
for cls, ap in sorted(results['per_class_ap'].items(), key=lambda x: -x[1]):
    bar = '█' * int(ap * 20)
    print(f'  {cls:<18} {ap:.3f}  {bar}')